In [1]:
import os
import pandas as pd
import textwrap
from plotnine import (
    ggplot, aes, geom_line, geom_point,
    ggtitle, xlab, ylab, theme,
    element_text
)
from plotnine.themes import theme_classic  # Cleaner theme for publication
from plotnine.scales import scale_color_hue # For distinct colors

In [2]:
# --- Load Excel file ---
def load_file(file_path, sheet_name):
    return pd.read_excel(file_path, sheet_name=sheet_name, engine="calamine")

df = (load_file(r"C:/Users/MTP24ME/Documents/CODE_PHD/meizeddin_PhD_code/Cycling/Data/Moha_NMC_200_73_Channel_12_Wb_1.xlsx", "Channel-12_1"))

In [3]:
# --- Filtering function ---
def filtering(df, num_of_filters, filter_column_names, ranges):
    if num_of_filters == len(filter_column_names) and num_of_filters == len(ranges):
        filtered_df = df.copy()
        for num in range(num_of_filters):
            filtered_df = filtered_df[filtered_df[filter_column_names[num]].isin(ranges[num])]
        return filtered_df
    else:
        raise ValueError("num_of_filters, filter_column_names, and ranges must have the same length")

In [1]:
def plotnine_ploting(df, multiple_graphs, legends, points_and_lines, column_filter_name, column_X, column_Y, plot_title, folderName):
    
    # 1. Prepare the data for plotnine
    # Ensure the "grouping" column is treated as a category for distinct colors
    if column_filter_name in df.columns:
        df[column_filter_name] = df[column_filter_name].astype('category')

    # 2. Set up the base plot and aesthetics
    if multiple_graphs:
        # Use the filter column for color and grouping
        gg = ggplot(df, aes(x=column_X, y=column_Y, color=column_filter_name, group=column_filter_name))
    else:
        # Just plot X vs Y as a single line
        gg = ggplot(df, aes(x=column_X, y=column_Y))

    # 4. Handle 'markevery=200'
    # We do this by creating a separate, sampled dataframe for the points
    markevery = 200
    if multiple_graphs:
        # Sample every 200th point *within each group*
        point_df = (df.groupby(column_filter_name, group_keys=True, observed= True)
                    .apply(lambda x: x.iloc[::markevery], include_groups=False)).reset_index()
    else:
        # Sample every 200th point from the whole dataframe
        point_df = df.iloc[::markevery]

    if points_and_lines == 2:
        # 3. Add lines
        # size=1.5 in plotnine is a thick line, similar to matplotlib's linewidth=4
        gg = gg + geom_line(size=1, linetype='solid') 
        # Add the points from the sampled data
        # size=2 in plotnine is similar to matplotlib's markersize=4
        gg = gg + geom_point(data=point_df, shape='o', size=6)
    elif points_and_lines == 1:
        # Add the points from the sampled data
        # size=2 in plotnine is similar to matplotlib's markersize=4
        gg = gg + geom_point(data=point_df, shape='o', size=6)
    else:
        # size=1.5 in plotnine is a thick line, similar to matplotlib's linewidth=4
        gg = gg + geom_line(size=0.8, linetype='solid') 
    
    
    # Use a qualitative (distinct) color palette. 
    # 'Paired' is colorblind-safe and good for many categories.
    if multiple_graphs:
        gg = gg + scale_color_hue(l=0.4, s=0.8)
        
    # 5. Add labels, title, and theme
    gg = (gg + ggtitle(plot_title) 
          + xlab(column_X) 
          + ylab(column_Y) 
          + theme_classic(base_size=14)  # <-- Set base font size
          + theme(
              plot_title=element_text(weight='bold', size=14), # Make title bold
              axis_title=element_text(weight='bold', size=14), # Bold axis labels
              legend_title=element_text(weight='bold', size=12),
              legend_position='right' # Or 'bottom', 'top', etc.
          ))
    
    # 6. Handle legend
    if not legends:
        gg = gg + theme(legend_position='none')

    # 7. Replicate your saving logic
    save_folder = r"Cycling/plots/" + folderName
    os.makedirs(save_folder, exist_ok=True)

    safe_title = plot_title.replace(" ", "_").replace("/", "_")
    save_path_png = os.path.join(save_folder, safe_title + ".png")
    save_path_pdf = os.path.join(save_folder, safe_title + ".pdf")
    
    # --- PNG Saving (for quick previews) ---
    if os.path.exists(save_path_png):
        response = input(f"⚠️ File '{save_path_png}' already exists. Overwrite? (y/n): ").strip().lower()
        if response == "y":
            gg.save(save_path_png, dpi=400, width=8, height=6, units="in")
            print(f"PNG saved to {save_path_png}")
    else:
        gg.save(save_path_png, dpi=400, width=8, height=6, units="in")
        print(f"PNG saved to {save_path_png}")

    # --- PNG Saving (for quick previews) ---
    if os.path.exists(save_path_pdf):
        response = input(f"⚠️ File '{save_path_png}' already exists. Overwrite? (y/n): ").strip().lower()
        if response == "y":
            gg.save(save_path_pdf, width=8, height=6, units="in")
            print(f"PNG saved to {save_path_pdf}")
    else:
        gg.save(save_path_pdf, width=8, height=6, units="in")
        print(f"PNG saved to {save_path_pdf}")

In [5]:
# --- Main Execution Block ---
# Note: This part is unchanged, except for calling the new function name

df = (load_file(r"C:/Users/MTP24ME/Documents/CODE_PHD/meizeddin_PhD_code/Cycling/Data/Moha_NMC_200_73_Channel_12_Wb_1.xlsx", "Channel-12_1"))
cycles = sorted(df["Cycle_Index"].unique())
    
selected_cycles = sorted(set(cycles[:2] + [c for c in cycles if c % 5 == 0] + cycles[-3:-1]))
print("Selected cycles:", selected_cycles)

filtered_df = filtering(df, 3, ["Step_Index", "Voltage(V)", "Cycle_Index"], [[3,7],[2.5], selected_cycles])
# Call the new function
plotnine_ploting(filtered_df, True, True, 1, "Cycle_Index", "Cycle_Index", "Specific Capacity (mAh/g)" , "Discharge Retention", "NMC_200_73")

filtered_df = filtering(df, 3, ["Step_Index", "Voltage(V)", "Cycle_Index"], [[2,6],[4.2], selected_cycles])
# Call the new function
plotnine_ploting(filtered_df, True, True, 1, "Cycle_Index", "Cycle_Index", "Specific Capacity (mAh/g)" , "Charge Retention", "NMC_200_73")

filtered_df = filtering(df, 2, ["Step_Index", "Cycle_Index"], [[2,6], selected_cycles])
# Call the new function
plotnine_ploting(filtered_df, True, True, 0, "Cycle_Index", "Specific Capacity (mAh/g)", "Voltage(V)", "Charging Voltage (V) vs Specific Capacity (mAh/g)", "NMC_200_73")

filtered_df = filtering(df, 2, ["Step_Index", "Cycle_Index"], [[3,7], selected_cycles])
# Call the new function
plotnine_ploting(filtered_df, True, True, 0, "Cycle_Index", "Specific Capacity (mAh/g)", "Voltage(V)", "Discharging Voltage (V) vs Specific Capacity (mAh/g)", "NMC_200_73")

Selected cycles: [1, 2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 51, 52]


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling/plots/NMC_200_73\Discharge_Retention.png


PNG saved to Cycling/plots/NMC_200_73\Discharge_Retention.png


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling/plots/NMC_200_73\Discharge_Retention.pdf


PNG saved to Cycling/plots/NMC_200_73\Discharge_Retention.pdf


...

PNG saved to Cycling/plots/NMC_200_73\Charging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling/plots/NMC_200_73\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).png


PNG saved to Cycling/plots/NMC_200_73\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).png


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling/plots/NMC_200_73\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf


PNG saved to Cycling/plots/NMC_200_73\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf
